In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-4o-mini'

In [16]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [01:02<00:00, 20.96s/it]


In [17]:
len(deals)

30

In [18]:
deals[10].describe()

'Title: Refurb Dell Latitude Laptop Sale at Dell Refurbished: Extra 40% off + shipping varies\nDetails: Dell Refurbished is offering a range of refurbished Latitude laptops, with prices starting at $215 after promo code "DELLSUMMER40" is applied. The extra 40% off coupon gets some of the lowest prices we\'ve seen this year. Some exclusions apply like Hot Deals. Each purchase includes the same limited hardware warranty Dell offers on new systems. Sale ends June 14, 2026. Shop Now at Dell Refurbished Store\nFeatures: Grades A and B cosmetic conditions available Processors ranging from 10th to 13th Gen Intel Core RAM options: 8GB, 16GB, 32GB, or 64GB Storage: 256GB, 512GB, or 1TB+ SSD Displays from 13.3" to 17" FHD, QHD+, or UHD+ Windows 10 Pro, Windows 11 Pro, or no OS options\nURL: https://www.dealnews.com/Refurb-Dell-Latitude-Laptop-Sale-at-Dell-Refurbished-Extra-40-off-shipping-varies/21839445.html?iref=rss-c39'

In [19]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [20]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [21]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: ESR 25W 3-in-1 Wireless Charger w/ MagSafe for $26 + free shipping
Details: Apply promo code "ESR2C571US85" to drop this ESR 3-in-1 MagSafe charger to $26, down from its regular price of $140. That beats the Amazon price today by almsot $20. The charger includes Apple-certified 15W MagSafe charging for iPhone, 5W fast charging for Apple Watch, and a built-in cooling fan that ke

In [22]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection)
results = response.choices[0].message.parsed
results

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


DealSelection(deals=[Deal(product_description='The ESR 25W 3-in-1 Wireless Charger is a highly efficient charging solution compatible with your iPhone and Apple Watch. Featuring Apple-certified 15W MagSafe charging for iPhones and 5W for Apple Watches, it also includes a built-in cooling fan to maintain optimal charging temperatures. The package comes with a 33W power adapter and a 5-ft USB-C cable, ensuring you have everything you need to power your devices effectively.', price=26.0, url='https://www.dealnews.com/products/ESR/ESR-25-W-3-in-1-Wireless-Charger-w-Mag-Safe/499109.html?iref=rss-c142'), Deal(product_description="The EcoFlow RAPID Pro Power Bank is a versatile and portable 10,000mAh battery solution that ensures you stay connected on the go. Designed for fast charging, it features multiple output options to accommodate various devices. Whether you're on a camping trip or just need a backup power source, this power bank is compact and reliable, fitting easily into your bag fo

In [23]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


The ESR 25W 3-in-1 Wireless Charger is a highly efficient charging solution compatible with your iPhone and Apple Watch. Featuring Apple-certified 15W MagSafe charging for iPhones and 5W for Apple Watches, it also includes a built-in cooling fan to maintain optimal charging temperatures. The package comes with a 33W power adapter and a 5-ft USB-C cable, ensuring you have everything you need to power your devices effectively.
26.0
https://www.dealnews.com/products/ESR/ESR-25-W-3-in-1-Wireless-Charger-w-Mag-Safe/499109.html?iref=rss-c142

The EcoFlow RAPID Pro Power Bank is a versatile and portable 10,000mAh battery solution that ensures you stay connected on the go. Designed for fast charging, it features multiple output options to accommodate various devices. Whether you're on a camping trip or just need a backup power source, this power bank is compact and reliable, fitting easily into your bag for convenient travel.
65.0
https://www.dealnews.com/Eco-Flow-RAPID-Pro-10-000-m-Ah-3-in-1-

In [3]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [2]:
from agents.scanner_agent import ScannerAgent

In [4]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [5]:
result

DealSelection(deals=[Deal(product_description='The ESR 25W 3-in-1 Wireless Charger is a versatile charging solution designed to accommodate Apple devices. It features Apple-certified 15W MagSafe charging for iPhone, along with a 5W fast charging option for the Apple Watch. The charger also includes a built-in CryoBoost cooling fan to ensure optimal charging temperatures. With a 33W power adapter and a 5-ft USB-C cable included, this sleek charger is perfect for keeping your devices powered up at home or on the go.', price=26.0, url='https://www.dealnews.com/products/ESR/ESR-25-W-3-in-1-Wireless-Charger-w-Mag-Safe/499109.html?iref=rss-c142'), Deal(product_description='The EcoFlow RAPID Pro 10,000mAh 3-in-1 Power Bank is an essential accessory for anyone needing portable power. This power bank provides a high-capacity battery that can charge your devices multiple times, ensuring you stay connected while traveling or during power outages. It features various output options for compatibili

In [6]:
load_dotenv(override=True)

True

In [7]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [8]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [9]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [11]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [13]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
03:03:36 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o-mini; provider = openai
03:03:38 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
